


* Extension neural network is a pattern recognition method found by M. H. Wang and C. P. Hung in 2003 to classify instances of data sets. Extension neural network is composed of artificial neural network and extension theory concepts. It uses the fast and adaptive learning capability of neural network and correlation estimation property of extension theory by calculating extension distance. 


 * I will be using the following datasets 
    * 1. kddcup99_csv.csv
    * 2. DataSetForPhishingVSBenignUrl.csv

In [ ]:
from sklearn.preprocessing import StandardScaler  
from sklearn.model_selection import train_test_split  
from sklearn.metrics import accuracy_score 
import numpy as np
import numpy.random as r 
import pandas as pd
import matplotlib.pyplot as plt 

from bokeh.plotting import figure, output_file, show
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter, Legend
from bokeh.io import output_notebook
output_notebook()

# Load dataset (adjust path as needed)
df = pd.read_csv('~/Desktop/Obfuscated-MalMem2022.csv')

# Inspect columns to help diagnose issues
print("DataFrame columns:", list(df.columns))

# Determine feature matrix X and label vector y in a robust way
if 'data' in df.columns:
    # 'data' column may contain arrays or scalars
    if isinstance(df.at[0, 'data'], (list, np.ndarray)):
        X = np.stack(df['data'].values)
    else:
        X = df['data'].values.reshape(-1, 1)
    # Look for a label column
    for candidate in ['target', 'label', 'Class', 'class']:
        if candidate in df.columns:
            y = df[candidate].values
            break
    else:
        raise ValueError("No label column found (expected 'target' or 'label' or 'Class').")

elif any(c in df.columns for c in ['target', 'label', 'Class', 'class']):
    # Use all columns except the target/label as features
    for candidate in ['target', 'label', 'Class', 'class']:
        if candidate in df.columns:
            target_col = candidate
            break
    y = df[target_col].values
    X_df = df.drop(columns=[target_col])
    # show which columns are non-numeric to help debugging
    non_numeric = [col for col in X_df.columns if not pd.api.types.is_numeric_dtype(X_df[col])]
    print("Non-numeric feature columns:", non_numeric)
    # One-hot encode categorical features
    X_df = pd.get_dummies(X_df)
    X = X_df.values

else:
    # Fallback: assume last column is the label and rest are features
    print("No 'data'/'target'/'label' columns found — assuming last column is the label.")
    X_df = df.iloc[:, :-1]
    y = df.iloc[:, -1].values
    non_numeric = [col for col in X_df.columns if not pd.api.types.is_numeric_dtype(X_df[col])]
    print("Non-numeric feature columns:", non_numeric)
    X_df = pd.get_dummies(X_df)
    X = X_df.values

print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

X_scale = StandardScaler()
X = X_scale.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# Convert labels to one-hot vectors (assumes labels are categorical integers or strings)
def convert_y_to_vect(y):
    classes = np.unique(y)
    K = classes.size
    class_to_index = {c:i for i,c in enumerate(classes)}
    y_vect = np.zeros((len(y), K))
    for i in range(len(y)):
        y_vect[i, class_to_index[y[i]]] = 1
    return y_vect



y_v_train = convert_y_to_vect(y_train)
y_v_test = convert_y_to_vect(y_test)

# -- remaining functions unchanged (activation, init, forward, backprop, training and plotting) --

def f(z):
    return 1 / (1 + np.exp(-z))


def f_deriv(z):
    return f(z) * (1 - f(z))

def setup_and_init_weights(nn_structure):
    W = {} 
    b = {}
    for l in range(1, len(nn_structure)):
        W[l] = r.random_sample((nn_structure[l], nn_structure[l-1])) 
        b[l] = r.random_sample((nn_structure[l],))
    return W, b

def init_tri_values(nn_structure):
    tri_W = {}
    tri_b = {}
    for l in range(1, len(nn_structure)):
        tri_W[l] = np.zeros((nn_structure[l], nn_structure[l-1]))
        tri_b[l] = np.zeros((nn_structure[l],))
    return tri_W, tri_b


def feed_forward(x, W, b):
    a = {1: x} 
    z = { } 
    for l in range(1, len(W) + 1): 
        node_in = a[l]
        z[l+1] = W[l].dot(node_in) + b[l]  
        a[l+1] = f(z[l+1]) 
    return a, z


def calculate_out_layer_delta(y, a_out, z_out):
    return -(y-a_out) * f_deriv(z_out) 


def calculate_hidden_delta(delta_plus_1, w_l, z_l):
    return np.dot(np.transpose(w_l), delta_plus_1) * f_deriv(z_l)


def predict_y(W, b, X, n_layers):
    N = X.shape[0]
    y = np.zeros((N,))
    for i in range(N):
        a, z = feed_forward(X[i, :], W, b)
        y[i] = np.argmax(a[n_layers])
    return y


def train_nn(nn_structure, X_train, y_train, X_test, y_test, iter_num=100, alpha=0.25):
    W, b = setup_and_init_weights(nn_structure)
    cnt = 0
    N = len(y_train)

    loss_train_seq = []
    loss_test_seq = []
    acc_train_seq = []
    acc_test_seq = []
    train_examples = len(y_train)
    test_examples = len(y_test)

    print('Starting gradient descent for {} epochs'.format(iter_num))
    while cnt < iter_num:
        print('Epoch {} of {}'.format(cnt+1, iter_num))
        tri_W, tri_b = init_tri_values(nn_structure)

        loss_train = 0.0
        loss_test = 0.0
        acc_train = 0.0
        acc_test = 0.0

        for i in range(N):
            delta = {}
            a, z = feed_forward(X_train[i, :], W, b)

            for l in range(len(nn_structure), 0, -1):
                if l == len(nn_structure):
                    delta[l] = calculate_out_layer_delta(y_train[i,:], a[l], z[l])
                    loss_train += np.linalg.norm((y_train[i,:]-a[l]))
                    if np.argmax(a[len(nn_structure)]) == np.argmax(y_v_train[i]):
                        acc_train += 1.0
                else:
                    if l > 1:
                        delta[l] = calculate_hidden_delta(delta[l+1], W[l], z[l])

                    tri_W[l] += np.dot(delta[l+1][:,np.newaxis], np.transpose(a[l][:,np.newaxis]))
                    tri_b[l] += delta[l+1]

        for l in range(len(nn_structure) - 1, 0, -1):
            W[l] += -alpha * (1.0/N * tri_W[l])
            b[l] += -alpha * (1.0/N * tri_b[l])

        for i in range(test_examples):
            a, z = feed_forward(X_test[i, :], W, b)
            loss_test += np.linalg.norm((y_test[i,:]-a[len(nn_structure)]))
            if np.argmax(a[len(nn_structure)]) == np.argmax(y_v_test[i]):
                acc_test += 1

        loss_train_seq.append(loss_train / train_examples)
        loss_test_seq.append(loss_test / test_examples)
        acc_train_seq.append(acc_train / train_examples)
        acc_test_seq.append(acc_test / test_examples)
        cnt += 1

    metrics = {
        'loss_train': loss_train_seq,
        'loss_test': loss_test_seq,
        'acc_train': acc_train_seq,
        'acc_test': acc_test_seq,
    }

    return W, b, metrics


nn_structure = [64, 30, 10]

epochs = 10

# Run training (use iter_num=epochs)
%time W, b, history = train_nn(nn_structure, X_train, y_v_train, X_test, y_v_test, iter_num=epochs, alpha=0.25)


def plot_results(loss_train_seq, loss_test_seq, acc_train_seq, acc_test_seq):
    print('The test set prediction accuracy is {}%'.format(acc_test_seq[-1] * 100))
    source = ColumnDataSource(data={
        'epoch'            : list(range(1, len(loss_test_seq) + 1)),
        'train_loss'    : loss_train_seq,
        'test_loss'        :  loss_test_seq,
    })

    p = figure(title='MNIST Loss: NN without CNN provided in HW', plot_width=400, plot_height=400)

    p.line(x='epoch', y='train_loss', color='blue', legend_label='Train Loss', source=source)
    p.line(x='epoch', y='test_loss', color='red', legend_label='Test Loss', source=source)
    p.legend.location = "top_right"
    p.xaxis.axis_label = 'Epochs'
    p.yaxis.axis_label = 'Loss' 

    p.add_tools(HoverTool(
        tooltips=[
                  ('Epochs', '@epoch{int}'),
                  ('Training Loss', '@train_loss{0.000 a}'),
                  ('Test Loss', '@test_loss{0.000 a}'),
        ],
        mode='mouse'
    ))

    show(p)

    source = ColumnDataSource(data={
        'epoch'            : list(range(1, len(acc_train_seq) + 1)),
        'train_acc'    : np.array(acc_train_seq),
        'test_acc'        :  np.array(acc_test_seq),
    })

    p = figure(title='MNIST Accuracy: NN without CNN provided in HW', plot_width=400, plot_height=400)

    p.line(x='epoch', y='train_acc', color='green', legend_label='Train Accuracy', source=source)
    p.line(x='epoch', y='test_acc', color='orange', legend_label='Test Accuracy', source=source)
    p.legend.location = "bottom_right"
    p.xaxis.axis_label = 'Epochs'
    p.yaxis.axis_label = 'Accuracy' 
    p.yaxis.formatter = NumeralTickFormatter(format='0 %')

    p.add_tools(HoverTool(
        tooltips=[
                  ('Epochs', '@epoch{int}'),
                  ('Training Accuracy', '@{train_acc}{%0.2f}'),
                  ('Test Accuracy', '@{test_acc}{%0.2f}'),
        ],
        mode='mouse'
    ))

    show(p)



loss_train_seq = history['loss_train']
loss_test_seq = history['loss_test']
acc_train_seq = history['acc_train']
acc_test_seq = history['acc_test']

# Plot
plot_results(loss_train_seq, loss_test_seq, acc_train_seq, acc_test_seq)

Loading BokehJS ...

DataFrame columns: ['Category', 'pslist.nproc', 'pslist.nppid', 'pslist.avg_threads', 'pslist.nprocs64bit', 'pslist.avg_handlers', 'dlllist.ndlls', 'dlllist.avg_dlls_per_proc', 'handles.nhandles', 'handles.avg_handles_per_proc', 'handles.nport', 'handles.nfile', 'handles.nevent', 'handles.ndesktop', 'handles.nkey', 'handles.nthread', 'handles.ndirectory', 'handles.nsemaphore', 'handles.ntimer', 'handles.nsection', 'handles.nmutant', 'ldrmodules.not_in_load', 'ldrmodules.not_in_init', 'ldrmodules.not_in_mem', 'ldrmodules.not_in_load_avg', 'ldrmodules.not_in_init_avg', 'ldrmodules.not_in_mem_avg', 'malfind.ninjections', 'malfind.commitCharge', 'malfind.protection', 'malfind.uniqueInjections', 'psxview.not_in_pslist', 'psxview.not_in_eprocess_pool', 'psxview.not_in_ethread_pool', 'psxview.not_in_pspcid_list', 'psxview.not_in_csrss_handles', 'psxview.not_in_session', 'psxview.not_in_deskthrd', 'psxview.not_in_pslist_false_avg', 'psxview.not_in_eprocess_pool_false_avg', 'psxview.not_in_ethr

In [6]:


plot_results(loss_train_seq, loss_test_seq, acc_train_seq, acc_test_seq)
     


NameError: name 'plot_results' is not defined